# Step 2 et 3 : Alignement avec Wikidata

- **Step 2** : relier nos entites a Wikidata avec `owl:sameAs`
- **Step 3** : relier nos predicats a Wikidata avec `owl:equivalentProperty`

In [1]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL, XSD
import json, re
from urllib.parse import quote

EX  = Namespace("http://musickg.example.org/resource/")
EXO = Namespace("http://musickg.example.org/ontology/")
DBO = Namespace("http://dbpedia.org/ontology/")
WD  = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")

CHEMIN = "C:/Users/sandy/OneDrive/Desktop/Cours A4/Web Datamining/TD4_Project/"
print(" OK")

 OK


### Chargement de la KB du Step 1

In [2]:
g = Graph()
g.bind("ex", EX); g.bind("exo", EXO); g.bind("dbo", DBO)
g.bind("wd", WD); g.bind("wdt", WDT); g.bind("owl", OWL)
g.bind("rdfs", RDFS); g.bind("xsd", XSD)

g.parse(CHEMIN + "private_kb.ttl", format="turtle")
print(f"KB chargee : {len(g)} triplets")

KB chargee : 1741 triplets


### Step 2 : alignement des entites avec Wikidata

Pour chaque artiste on ajoute `owl:sameAs` vers son identifiant Wikidata.
3 strategies selon le niveau d'ambiguïté du nom :
- `exact` (0.99) : nom unique dans Wikidata
- `alias` (0.97) : nom avec caractere special normalise
- `context` (0.95-0.97) : nom ambigu, verifie par type et pays

In [3]:
ALIGNEMENT_ENTITES = {
    # Francophones
    "Daft Punk":         ("Q184803",  0.99, "exact"),
    "Stromae":           ("Q193337",  0.99, "exact"),
    "Celine Dion":       ("Q3083",    0.97, "alias"),    # accent manquant sur Céline
    "Edith Piaf":        ("Q1631",    0.97, "alias"),    # accent manquant sur Édith
    "Serge Gainsbourg":  ("Q232969",  0.99, "exact"),
    "Charles Aznavour":  ("Q170749",  0.99, "exact"),
    "Jacques Brel":      ("Q192669",  0.99, "exact"),
    "MC Solaar":         ("Q520922",  0.99, "exact"),
    "Air":               ("Q389488",  0.97, "context"),  # nom tres commun
    "Mylene Farmer":     ("Q235196",  0.97, "alias"),    # accent manquant sur Mylène
    # Anglophones
    "The Beatles":       ("Q1299",    0.99, "exact"),
    "Pink Floyd":        ("Q2306",    0.99, "exact"),
    "Led Zeppelin":      ("Q2331",    0.99, "exact"),
    "Radiohead":         ("Q128309",  0.99, "exact"),
    "David Bowie":       ("Q5383",    0.99, "exact"),
    "Michael Jackson":   ("Q2831",    0.99, "exact"),
    "Bob Dylan":         ("Q392",     0.99, "exact"),
    "Rolling Stones":    ("Q11036",   0.99, "exact"),
    "Queen":             ("Q15862",   0.97, "context"),  # nom ambigu
    "Nirvana":           ("Q11649",   0.97, "context"),  # nom ambigu
    "Beyonce":           ("Q83405",   0.97, "alias"),    # accent manquant sur Beyoncé
    "Eminem":            ("Q1199",    0.99, "exact"),
    "Jay-Z":             ("Q170930",  0.99, "exact"),
    "Kanye West":        ("Q44088",   0.99, "exact"),
    "Adele":             ("Q81771",   0.97, "context"),  # prenom commun
    "Coldplay":          ("Q45188",   0.99, "exact"),
    "Arctic Monkeys":    ("Q134541",  0.99, "exact"),
    "Tame Impala":       ("Q869612",  0.99, "exact"),
    "Amy Winehouse":     ("Q131272",  0.99, "exact"),
    "Miles Davis":       ("Q93341",   0.99, "exact"),
    "Nina Simone":       ("Q128439",  0.99, "exact"),
    "Elvis Presley":     ("Q303",     0.99, "exact"),
    "Bob Marley":        ("Q76",      0.99, "exact"),
    "Aretha Franklin":   ("Q5990",    0.99, "exact"),
    "Frank Sinatra":     ("Q40912",   0.99, "exact"),
    "Kendrick Lamar":    ("Q205303",  0.99, "exact"),
    "Drake":             ("Q33777",   0.97, "context"),  # prenom commun
    "Rihanna":           ("Q36153",   0.99, "exact"),
    "Whitney Houston":   ("Q37079",   0.99, "exact"),
    "Massive Attack":    ("Q183504",  0.99, "exact"),
    "Portishead":        ("Q217294",  0.99, "exact"),
    "Gorillaz":          ("Q259732",  0.99, "exact"),
    "Joy Division":      ("Q183412",  0.99, "exact"),
    "Aphex Twin":        ("Q208638",  0.99, "exact"),
    "The Strokes":       ("Q183048",  0.99, "exact"),
    "LCD Soundsystem":   ("Q1352505", 0.99, "exact"),
    "Bjork":             ("Q47159",   0.97, "alias"),    # accent manquant sur Björk
    "Tupac Shakur":      ("Q155818",  0.99, "exact"),
}

print(f"{len(ALIGNEMENT_ENTITES)} artistes a aligner.")

48 artistes a aligner.


In [4]:
def safe_uri(name):
    clean = re.sub(r"[^\w\s\-]", "", name).strip()
    clean = re.sub(r"\s+", "_", clean)
    return EX[quote(clean, safe="_-")]

triplets_avant = len(g)
table_alignement = []

for nom, (qid, confiance, strategie) in ALIGNEMENT_ENTITES.items():
    uri_prive    = safe_uri(nom)
    uri_wikidata = WD[qid]

    g.add((uri_prive,    OWL.sameAs, uri_wikidata))
    g.add((uri_wikidata, OWL.sameAs, uri_prive))
    g.add((uri_prive, EXO.alignmentConfidence, Literal(confiance, datatype=XSD.decimal)))
    g.add((uri_prive, EXO.alignmentStrategy,   Literal(strategie)))

    table_alignement.append({
        "entite_privee": f"ex:{nom.replace(' ', '_')}",
        "uri_wikidata":  f"wd:{qid}",
        "confiance":     confiance,
        "strategie":     strategie
    })
    print(f"  {nom:<25} -> wd:{qid}  (conf={confiance}, strat={strategie})")

print()
print(f"Triplets ajoutes : {len(g) - triplets_avant}")
print(f"Total            : {len(g)}")

  Daft Punk                 -> wd:Q184803  (conf=0.99, strat=exact)
  Stromae                   -> wd:Q193337  (conf=0.99, strat=exact)
  Celine Dion               -> wd:Q3083  (conf=0.97, strat=alias)
  Edith Piaf                -> wd:Q1631  (conf=0.97, strat=alias)
  Serge Gainsbourg          -> wd:Q232969  (conf=0.99, strat=exact)
  Charles Aznavour          -> wd:Q170749  (conf=0.99, strat=exact)
  Jacques Brel              -> wd:Q192669  (conf=0.99, strat=exact)
  MC Solaar                 -> wd:Q520922  (conf=0.99, strat=exact)
  Air                       -> wd:Q389488  (conf=0.97, strat=context)
  Mylene Farmer             -> wd:Q235196  (conf=0.97, strat=alias)
  The Beatles               -> wd:Q1299  (conf=0.99, strat=exact)
  Pink Floyd                -> wd:Q2306  (conf=0.99, strat=exact)
  Led Zeppelin              -> wd:Q2331  (conf=0.99, strat=exact)
  Radiohead                 -> wd:Q128309  (conf=0.99, strat=exact)
  David Bowie               -> wd:Q5383  (conf=0.99, str

### Ce qu'on observe

Sur 48 artistes, 36 ont obtenu un score de confiance de 0.99 (stratégie "exact") car
leur nom est unique dans Wikidata et ne prête pas à confusion.

9 artistes ont nécessité une attention particulière :
- **Alias** (0.97) : Céline Dion, Édith Piaf, Mylène Farmer, Beyoncé, Björk ont des
  accents absents de nos URIs. On a cherché la version sans accent et vérifié le Q-ID.
- **Context** (0.97) : Queen, Nirvana, Adele, Air, Drake ont des noms ambigus.
  Pour chacun on a confirmé le type (MusicGroup/Person) et le pays d'origine.

### Step 3 : alignement des predicats

- `owl:equivalentProperty` : sens identique
- `rdfs:subPropertyOf` : notre predicat est plus specifique

In [5]:
ALIGNEMENT_PREDICATS = [
    (EXO.hasGenre,         "P136", "genre",                 OWL.equivalentProperty,  0.99, "match exact"),
    (EXO.signedTo,         "P264", "record label",          OWL.equivalentProperty,  0.98, "match exact"),
    (EXO.influencedBy,     "P737", "influenced by",         OWL.equivalentProperty,  0.99, "match exact"),
    (EXO.memberOf,         "P463", "member of",             OWL.equivalentProperty,  0.97, "match exact"),
    (EXO.releaseYear,      "P577", "publication date",      OWL.equivalentProperty,  0.96, "match exact"),
    (EXO.releasedAlbum,    "P358", "discography",           OWL.equivalentProperty,  0.85, "P358 plus large mais le plus proche"),
    (EXO.originCountry,    "P495", "country of origin",     OWL.equivalentProperty,  0.90, "P495 pour groupes, P27 pour individus"),
    (EXO.activeFrom,       "P571", "inception",             OWL.equivalentProperty,  0.85, "P571 pour groupes, P569 pour individus"),
    (EXO.activeTo,         "P576", "dissolved date",        OWL.equivalentProperty,  0.85, "P576 pour groupes, P570 pour individus"),
    (EXO.hasMember,        "P527", "has part",              RDFS.subPropertyOf,      0.88, "hasMember est plus specifique que hasPart"),
    (EXO.collaboratedWith, "P1327","partner in business",   RDFS.subPropertyOf,      0.70, "P1327 trop generique, le notre est musical"),
]

resultats_predicats = []
for pred_uri, prop_wd, label_wd, relation_rdf, confiance, note in ALIGNEMENT_PREDICATS:
    wd_uri = WDT[prop_wd]
    g.add((pred_uri, relation_rdf, wd_uri))
    g.add((pred_uri, EXO.alignmentConfidence, Literal(confiance, datatype=XSD.decimal)))
    g.add((pred_uri, RDFS.comment, Literal(note, lang="fr")))

    type_rel = "owl:equivalentProperty" if relation_rdf == OWL.equivalentProperty else "rdfs:subPropertyOf"
    symbole  = "=" if relation_rdf == OWL.equivalentProperty else "<"
    nom_pred = str(pred_uri).split("/")[-1]

    print(f"  exo:{nom_pred:<22} {symbole} wdt:{prop_wd}  ({label_wd})")
    print(f"    conf={confiance}  |  {note}")
    print()

    resultats_predicats.append({
        "predicat_prive": f"exo:{nom_pred}",
        "prop_wikidata":  f"wdt:{prop_wd}",
        "label_wikidata": label_wd,
        "relation":       type_rel,
        "confiance":      confiance,
        "note":           note
    })

print(f"Total triplets apres Step 3 : {len(g)}")

  exo:hasGenre               = wdt:P136  (genre)
    conf=0.99  |  match exact

  exo:signedTo               = wdt:P264  (record label)
    conf=0.98  |  match exact

  exo:influencedBy           = wdt:P737  (influenced by)
    conf=0.99  |  match exact

  exo:memberOf               = wdt:P463  (member of)
    conf=0.97  |  match exact

  exo:releaseYear            = wdt:P577  (publication date)
    conf=0.96  |  match exact

  exo:releasedAlbum          = wdt:P358  (discography)
    conf=0.85  |  P358 plus large mais le plus proche

  exo:originCountry          = wdt:P495  (country of origin)
    conf=0.9  |  P495 pour groupes, P27 pour individus

  exo:activeFrom             = wdt:P571  (inception)
    conf=0.85  |  P571 pour groupes, P569 pour individus

  exo:activeTo               = wdt:P576  (dissolved date)
    conf=0.85  |  P576 pour groupes, P570 pour individus

  exo:hasMember              < wdt:P527  (has part)
    conf=0.88  |  hasMember est plus specifique que hasPart

  e

### Ce qu'on observe

Sur 11 prédicats alignés, 9 utilisent `owl:equivalentProperty` (sens identique)
et 2 utilisent `rdfs:subPropertyOf` (notre prédicat est plus spécifique).

Les deux cas de sous-propriété :
- **hasMember < wdt:P527** : Wikidata utilise "has part" de façon générale.
  Notre prédicat est restreint aux membres humains d'un groupe musical.
- **collaboratedWith < wdt:P1327** : aucun prédicat Wikidata ne modélise
  exactement une collaboration musicale. C'est une limite de Wikidata.

3 prédicats ont une confiance plus faible (0.85) car Wikidata distingue
plusieurs propriétés selon le type d'entité (groupe vs personne).

### Statistiques finales

In [6]:
print("STATISTIQUES APRES ALIGNEMENT")
print(f"Triplets totaux        : {len(g)}")
print(f"Entites uniques        : {len(set(g.subjects()))}")
print(f"Predicats uniques      : {len(set(g.predicates()))}")
print(f"Liens owl:sameAs       : {len(list(g.triples((None, OWL.sameAs, None))))}")
print(f"Entites alignees       : {len(ALIGNEMENT_ENTITES)}")
print(f"Predicats alignes      : {len(ALIGNEMENT_PREDICATS)}")

STATISTIQUES APRES ALIGNEMENT
Triplets totaux        : 1918
Entites uniques        : 426
Predicats uniques      : 24
Liens owl:sameAs       : 96
Entites alignees       : 48
Predicats alignes      : 11


### Bilan

Les liens `owl:sameAs` ajoutés dans les deux sens serviront de points d'entrée
pour l'expansion SPARQL au Step 4 : pour chaque artiste on connaît maintenant
son Q-ID Wikidata, ce qui permet d'interroger directement la base ouverte.

### Sauvegarde

In [7]:
g.serialize(CHEMIN + "aligned_kb.ttl", format="turtle")
print("Fichier sauvegarde : aligned_kb.ttl")

g.serialize(CHEMIN + "aligned_kb.nt", format="ntriples")
print("Fichier sauvegarde : aligned_kb.nt")

with open(CHEMIN + "alignment_table.json", "w", encoding="utf-8") as f:
    json.dump({
        "alignement_entites":   table_alignement,
        "alignement_predicats": resultats_predicats
    }, f, indent=2, ensure_ascii=False)
print("Fichier sauvegarde : alignment_table.json")

Fichier sauvegarde : aligned_kb.ttl
Fichier sauvegarde : aligned_kb.nt
Fichier sauvegarde : alignment_table.json


C:\Users\sandy\anaconda3\envs\rstudio\lib\site-packages\rdflib\plugins\serializers\nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  "NTSerializer always uses UTF-8 encoding. "
